## 1. Environment Setup and Library Imports

In [ ]:
import os
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# -------------------------------------------------------------------------
# 1. Path resolution to access root directory and 'src' modules
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import local custom modules
from src.pkldataset import PKLDataset, condition_to_id
from src.helpers import set_seed, get_model, train_model

# -------------------------------------------------------------------------
# 2. Hardware and Training Hyperparameters
# -------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()
NUM_CLASSES_MATERIAL = 2  # Binary condition classification: HC=0, DC=1
NUM_EPOCHS = 35           # Standard convergence limit for binary regime
MODALITY = "FUSION"       # Early fusion modality: Force (RP) + Acceleration (AZ)
INPUT_CHANNELS = 2        # 2 synchronized input channels

# 10 random seeds for statistical evaluation
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# -------------------------------------------------------------------------
# 3. Experimental Matrix Definition
# -------------------------------------------------------------------------
source_thickness = "T197"
thickness_list = ["T185", "T188", "T191", "T194", "T197"]

subset_sizes = [20, 50, 100, 200, 300, 400, 500]
subset_names = [f"train_{s}" for s in subset_sizes]

# -------------------------------------------------------------------------
# 4. Folder Path Verification (RP and AZ folders for HC and DC across thicknesses)
# -------------------------------------------------------------------------
print("==================================================================")
print(f"STEP 1: Verifying folder paths for Material Classification ({MODALITY}: RP + AZ)...")
print("==================================================================")

all_paths_ok = True
for t_name in thickness_list:
    for regime in ["HC", "DC"]:
        for sub_mod in ["RP", "AZ"]:
            folder_path = DATA_DIR / regime / t_name / sub_mod
            exists = folder_path.exists()
            status = "FOUND" if exists else "MISSING"
            if not exists:
                all_paths_ok = False
            print(f"[{regime} {t_name} {sub_mod}] {folder_path} -> {status}")

print(f"\nPath verification status: {'ALL PATHS OK' if all_paths_ok else 'WARNING: SOME PATHS ARE MISSING'}")
print(f"Device: {device}")
print(f"Source Training Thickness: {source_thickness}")
print(f"Evaluation Thicknesses: {thickness_list}")
print(f"Training Subsets: {subset_names}")
print(f"Input Channels: {INPUT_CHANNELS}")
print(f"Total Random Seeds: {len(seeds)}")

## 2. Configuration (Fusion)

In [ ]:
# -------------------------------------------------------------------------
# STEP 2: Register Dataset References for Fusion (RP + AZ)
# -------------------------------------------------------------------------
print("==================================================================")
print(f"STEP 2: Registering dual-modality dataset references ({MODALITY})...")
print("==================================================================")

# Store paired references: {"RP": ds_rp, "AZ": ds_az} for each thickness
thickness_datasets = {}

for t_name in thickness_list:
    rp_paths = [
        str(DATA_DIR / "HC" / t_name / "RP"),
        str(DATA_DIR / "DC" / t_name / "RP"),
    ]
    az_paths = [
        str(DATA_DIR / "HC" / t_name / "AZ"),
        str(DATA_DIR / "DC" / t_name / "AZ"),
    ]
    
    ds_rp = PKLDataset(folder_path=rp_paths, modality="single", task="condition")
    ds_az = PKLDataset(folder_path=az_paths, modality="single", task="condition")
    
    assert len(ds_rp) == len(ds_az), f"Length mismatch in {t_name}: RP ({len(ds_rp)}) vs AZ ({len(ds_az)})"
    
    thickness_datasets[t_name] = {
        "RP": ds_rp,
        "AZ": ds_az
    }
    print(f"[{t_name}] Dual datasets registered: {len(ds_rp)} synchronized samples (HC + DC)")

print("\nFusion dataset references initialized successfully.")

## 3. Validation Preloading - Fusion

In [ ]:
# -------------------------------------------------------------------------
# STEP 3: Preload Both Sensors into GPU VRAM as 2-Channel Tensors [N, 2, L]
# -------------------------------------------------------------------------
print("==================================================================")
print(f"STEP 3: Preloading & Concatenating {MODALITY} (RP + AZ) into GPU memory...")
print("==================================================================")

cached_thickness_data = {}

for t_name, dsets in thickness_datasets.items():
    loader_rp = DataLoader(dsets["RP"], batch_size=512, shuffle=False, num_workers=4, pin_memory=True if device.type == "cuda" else False)
    loader_az = DataLoader(dsets["AZ"], batch_size=512, shuffle=False, num_workers=4, pin_memory=True if device.type == "cuda" else False)
    
    xs_rp, ys_rp = [], []
    for x, y in loader_rp:
        xs_rp.append(x)
        ys_rp.append(y)
        
    xs_az, _ = [], []
    for x, _ in loader_az:
        xs_az.append(x)
        
    tensor_rp = torch.cat(xs_rp, dim=0).to(device)  # Shape: [N, 1, L]
    tensor_az = torch.cat(xs_az, dim=0).to(device)  # Shape: [N, 1, L]
    targets = torch.cat(ys_rp, dim=0).view(-1).long().to(device)
    
    # Early fusion: channel-wise concatenation along dimension 1 -> [N, 2, L]
    fused_inputs = torch.cat([tensor_rp, tensor_az], dim=1)
    
    cached_thickness_data[t_name] = {
        "inputs": fused_inputs,
        "targets": targets
    }
    
    n_samples = fused_inputs.shape[0]
    n_hc = (targets == condition_to_id['HC']).sum().item()
    n_dc = (targets == condition_to_id['DC']).sum().item()
    mem_mb = fused_inputs.element_size() * fused_inputs.nelement() / 1e6
    
    print(f"[{t_name}] Cached in {device}: {n_samples} samples | Shape: {tuple(fused_inputs.shape)} ({mem_mb:.1f} MB) | HC: {n_hc}, DC: {n_dc}")

print(f"\nAll {MODALITY} dual-channel datasets successfully cached in memory. Zero disk I/O remaining.")

## 4. Training and Evaluation - Fusion

In [ ]:
from torch.utils.data import TensorDataset

# -------------------------------------------------------------------------
# 1. Fast In-Memory Evaluation Helper
# -------------------------------------------------------------------------
def fast_eval_condition(model, inputs_tensor, targets_tensor, batch_size=512, device="cuda"):
    """
    Evaluates binary classification accuracy directly on 2-channel in-memory tensors.
    """
    model.eval()
    correct = 0
    total = targets_tensor.size(0)
    
    with torch.no_grad():
        for i in range(0, total, batch_size):
            x = inputs_tensor[i:i + batch_size].to(device)
            y = targets_tensor[i:i + batch_size].to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            
    return 100.0 * correct / total

# -------------------------------------------------------------------------
# 2. Extract Training Pools from Source Thickness (T197)
# -------------------------------------------------------------------------
X_src = cached_thickness_data[source_thickness]["inputs"]
Y_src = cached_thickness_data[source_thickness]["targets"]

idx_hc_src = (Y_src == condition_to_id['HC']).nonzero(as_tuple=True)[0].cpu().numpy()
idx_dc_src = (Y_src == condition_to_id['DC']).nonzero(as_tuple=True)[0].cpu().numpy()

# Results container: {subset_name: {thickness: [acc_seed_1, ...]}}
results_matrix = {s_name: {t: [] for t in thickness_list} for s_name in subset_names}

# -------------------------------------------------------------------------
# 3. Optimized Training & Cross-Thickness Loop Across 10 Seeds
# -------------------------------------------------------------------------
print("==================================================================")
print(f"STEP 4: Training on {source_thickness} Subsets & Cross-Thickness Evaluation ({MODALITY})...")
print("==================================================================")

for seed in seeds:
    print(f"\n==========================================")
    print(f">>> Running Material Classification ({MODALITY}) for Seed: {seed}")
    print(f"==========================================")
    set_seed(seed)
    
    np.random.seed(seed)
    shuffled_hc = np.random.permutation(idx_hc_src)
    shuffled_dc = np.random.permutation(idx_dc_src)
    
    for s_size, s_name in zip(subset_sizes, subset_names):
        # Balanced 50% HC and 50% DC
        n_per_class = s_size // 2
        selected_idx = np.concatenate([
            shuffled_hc[:n_per_class],
            shuffled_dc[:n_per_class]
        ])
        np.random.shuffle(selected_idx)
        
        train_x = X_src[selected_idx]
        train_y = Y_src[selected_idx]
        
        curr_batch_size = min(64, s_size)
        train_loader = DataLoader(
            TensorDataset(train_x, train_y),
            batch_size=curr_batch_size,
            shuffle=True
        )
        
        # 1D-CNN instance initialized with 2 input channels (RP + AZ)
        model = get_model(input_channels=INPUT_CHANNELS, num_classes=2).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)
        
        # Train model
        model = train_model(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            num_epochs=NUM_EPOCHS,
            device=device
        )
        
        # Direct in-memory evaluation across all thickness sets
        for t_name in thickness_list:
            acc = fast_eval_condition(
                model=model,
                inputs_tensor=cached_thickness_data[t_name]["inputs"],
                targets_tensor=cached_thickness_data[t_name]["targets"],
                batch_size=512,
                device=device
            )
            results_matrix[s_name][t_name].append(acc)
            
    print(f"[Seed {seed}] Completed evaluation across all subsets.")

print(f"\nStep 4 completed successfully. Metrics recorded in results_matrix.")

## 5. Summary of Results - Fusion

In [ ]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# STEP 5: Compile Final Summary Table and Export (FUSION)
# -------------------------------------------------------------------------
print("==================================================================")
print(f"STEP 5: Compiling Consolidated Results Table ({MODALITY})...")
print("==================================================================")

summary_rows = []

for s_name in subset_names:
    row = {f"Training Subset ({source_thickness})": s_name}
    for t_name in thickness_list:
        scores = results_matrix[s_name][t_name]
        mean_acc = np.mean(scores)
        std_acc = np.std(scores)
        row[f"Test {t_name} (%)"] = f"{mean_acc:.2f} ± {std_acc:.2f}"
    summary_rows.append(row)

df_condition_fusion_matrix = pd.DataFrame(summary_rows)

# Strict column ordering
ordered_cols = [f"Training Subset ({source_thickness})"] + [f"Test {t} (%)" for t in thickness_list]
df_condition_fusion_matrix = df_condition_fusion_matrix[ordered_cols]

print(f"\nConsolidated Material Condition Classification Results ({MODALITY}) [Mean ± Std over {len(seeds)} seeds]:")
display(df_condition_fusion_matrix)

# Concise export filename
output_csv = f"mat_cond_{MODALITY.lower()}_eval.csv"
df_condition_fusion_matrix.to_csv(output_csv, index=False)
print(f"\nFinal results successfully exported to: {output_csv}")